# Hasil eksekusi Colab: 03b_naive_bayes_manual

Dijalankan di VM Colab GPU T4 (sesi `d920cd`). Sel di bawah adalah kode notebook asli beserta outputnya.


In [ ]:
import pandas as pd
import numpy as np
import math
import random

random.seed(42)
np.random.seed(42)

# Dataset bersih hasil notebook 02_bersihkan_data, diambil dari repositori supaya
# isinya sama persis dengan yang dipakai notebook lain.
URL_BERSIH = ("https://raw.githubusercontent.com/ravi-arnan/machine-learning/"
              "main/artifacts/stroke_bersih.csv")

KOLOM = ["gender", "age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"]
df = pd.read_csv(URL_BERSIH)[KOLOM]

# Fitur untuk model. gender tidak dipakai sebagai fitur, hanya untuk skor medis di bagian 5.
fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

print("Ukuran dataset bersih:", df.shape)
print("BMI kosong:", df["bmi"].isna().sum())
print("5 baris pertama:")
df.head()

Ukuran dataset bersih: (5109, 7)
BMI kosong: 0
5 baris pertama:


,gender,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
0,Male,67.0,0,1,228.69,36.60000,1
1,Female,61.0,0,0,202.21,31.84859,1
2,Male,80.0,0,1,105.92,32.50000,1
3,Female,49.0,0,0,171.23,34.40000,1
4,Female,79.0,1,0,174.12,24.00000,1


In [ ]:
def train_test_split_manual(data, test_size=0.2, seed=42):
    """Bagi baris data jadi train dan test secara manual (80:20)."""
    random.seed(seed)
    n = len(data)
    n_test = int(n * test_size)

    indices = list(range(n))
    random.shuffle(indices)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return data.iloc[train_idx], data.iloc[test_idx]


latih, uji = train_test_split_manual(df, test_size=0.2, seed=42)

X_train, y_train = latih[fitur].values, latih["stroke"].values
X_test, y_test = uji[fitur].values, uji["stroke"].values

print(f"Train: {len(latih)} baris")
print(f"Test:  {len(uji)} baris")
print(f"Proporsi stroke di train: {y_train.mean():.3f}")
print(f"Proporsi stroke di test:  {y_test.mean():.3f}")

Train: 4088 baris
Test:  1021 baris
Proporsi stroke di train: 0.046
Proporsi stroke di test:  0.060


In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    """Hitung TP, TN, FP, FN manual."""
    tp = tn = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            tp += 1
        elif t == 0 and p == 0:
            tn += 1
        elif t == 0 and p == 1:
            fp += 1
        elif t == 1 and p == 0:
            fn += 1
    return tp, tn, fp, fn


def hitung_metrik(y_true, y_pred):
    """Hitung akurasi, presisi, recall, F1 manual."""
    tp, tn, fp, fn = confusion_matrix_manual(y_true, y_pred)

    akurasi = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    presisi = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * presisi * recall / (presisi + recall) if (presisi + recall) > 0 else 0

    return {
        "akurasi": round(akurasi, 4),
        "presisi": round(presisi, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

In [ ]:
def gaussian_pdf(x, mean, std):
    """Hitung probabilitas Gaussian."""
    if std == 0:
        return 1.0 if x == mean else 0.0001
    eksponen = math.exp(-((x - mean) ** 2) / (2 * std ** 2))
    return (1 / (math.sqrt(2 * math.pi) * std)) * eksponen


class NaiveBayesManual:
    """Gaussian Naive Bayes buatan sendiri."""

    def fit(self, X, y):
        n, m = X.shape
        self.kelas = sorted(set(y))
        self.prior = {}
        self.mean = {}
        self.std = {}

        for c in self.kelas:
            Xc = X[y == c]
            self.prior[c] = len(Xc) / n
            self.mean[c] = Xc.mean(axis=0)
            self.std[c] = Xc.std(axis=0)

    def predict(self, X):
        y_pred = []
        for titik in X:
            prob_kelas = {}
            for c in self.kelas:
                prob = math.log(self.prior[c])
                for i in range(len(titik)):
                    prob += math.log(gaussian_pdf(titik[i], self.mean[c][i], self.std[c][i]) + 1e-10)
                prob_kelas[c] = prob
            y_pred.append(max(prob_kelas, key=prob_kelas.get))
        return np.array(y_pred)


nb = NaiveBayesManual()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("Naive Bayes selesai.")

Naive Bayes selesai.


In [ ]:
metrik_nb = hitung_metrik(y_test, y_pred_nb)

print("=" * 50)
print("HASIL NAIVE BAYES")
print("=" * 50)
print(f"  Akurasi:  {metrik_nb['akurasi']}")
print(f"  Presisi:  {metrik_nb['presisi']}")
print(f"  Recall:   {metrik_nb['recall']}")
print(f"  F1-score: {metrik_nb['f1']}")
print()
print("Confusion Matrix:")
print(f"  TP={metrik_nb['tp']}  FP={metrik_nb['fp']}")
print(f"  FN={metrik_nb['fn']}  TN={metrik_nb['tn']}")

HASIL NAIVE BAYES
  Akurasi:  0.8727
  Presisi:  0.2149
  Recall:   0.4262
  F1-score: 0.2857

Confusion Matrix:
  TP=26  FP=95
  FN=35  TN=865


In [ ]:
nama_fitur = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]

for c in nb.kelas:
    label = "stroke" if c == 1 else "tidak stroke"
    print(f"Kelas {c} ({label}), prior {nb.prior[c]:.4f}")
    for i, nama in enumerate(nama_fitur):
        print(f"  {nama:18s} mean={nb.mean[c][i]:8.2f}  std={nb.std[c][i]:8.2f}")
    print()

Kelas 0 (tidak stroke), prior 0.9540
  age                mean=   41.82  std=   22.42
  hypertension       mean=    0.09  std=    0.28
  heart_disease      mean=    0.05  std=    0.21
  avg_glucose_level  mean=  104.47  std=   43.39
  bmi                mean=   28.78  std=    7.88

Kelas 1 (stroke), prior 0.0460
  age                mean=   68.26  std=   12.05
  hypertension       mean=    0.26  std=    0.44
  heart_disease      mean=    0.19  std=    0.39
  avg_glucose_level  mean=  132.51  std=   61.41
  bmi                mean=   30.19  std=    5.81



In [ ]:
def skor_cha2ds2_vasc(baris):
    """CHA2DS2-VASc versi adaptasi. Penjelasan rumusnya ada di sel atas."""
    skor = 0
    if baris["heart_disease"] == 1:          # C, proksi gagal jantung
        skor += 1
    if baris["hypertension"] == 1:           # H
        skor += 1
    if baris["age"] >= 75:                   # A2
        skor += 2
    elif baris["age"] >= 65:                 # A
        skor += 1
    if baris["avg_glucose_level"] >= 126:    # D, ambang diabetes menurut ADA
        skor += 1
    if baris["gender"] == "Female":          # Sc
        skor += 1
    # S2 (riwayat stroke) sengaja tidak dipakai: itu label yang sedang diprediksi.
    # V (penyakit vaskular) tidak tersedia di dataset.
    return skor


skor = df.apply(skor_cha2ds2_vasc, axis=1)
skor_uji = skor.loc[uji.index].values

print("Sebaran skor pada data uji:")
print(pd.Series(skor_uji).value_counts().sort_index().to_string())

Sebaran skor pada data uji:
0    252
1    462
2    160
3     89
4     45
5     13


In [ ]:
baris_skor = []
for ambang in [1, 2, 3, 4]:
    pred = (skor_uji >= ambang).astype(int)
    m = hitung_metrik(y_test, pred)
    baris_skor.append({
        "Aturan": f"skor >= {ambang}",
        "akurasi": m["akurasi"],
        "presisi": m["presisi"],
        "recall": m["recall"],
        "f1": m["f1"],
        "ditandai": int(pred.sum()),
    })

pd.DataFrame(baris_skor)

,Aturan,akurasi,presisi,recall,f1,ditandai
0,skor >= 1,0.2987,0.0741,0.9344,0.1373,769
1,skor >= 2,0.7218,0.1368,0.6885,0.2283,307
2,skor >= 3,0.8570,0.2109,0.5082,0.2981,147
3,skor >= 4,0.9167,0.2931,0.2787,0.2857,58
